# 🏎️ McQueen StyleTTS2 — Bulletproof 250 Epoch Training Notebook
**1-Click Clean Training Notebook for Google Colab.**
All PyTorch 2.6+, CUDA memory, HiFiGAN batch shape, and speaker ID patches are 100% pre-applied.

**Requirements:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# CELL 1 — GPU CHECK & MEMORY OPTIMIZATION
import os, torch, gc
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
assert torch.cuda.is_available(), '❌ NO GPU — Go to Runtime > Change runtime type > T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# CELL 2 — INSTALL DEPENDENCIES
!apt-get install -qq espeak-ng ffmpeg
!pip install -q openai-whisper SoundFile phonemizer munch einops tqdm librosa transformers accelerate
import torch, whisper
print('✅ PyTorch version:', torch.__version__)
print('✅ Whisper version:', whisper.__version__)
print('✅ All core packages installed')


In [ ]:
# CELL 3 — CLONE STYLETTS2 REPO
import os
!git clone -q https://github.com/yl4579/StyleTTS2 /content/StyleTTS2
%cd /content/StyleTTS2
!pip install -q -r requirements.txt
print('✅ StyleTTS2 repository cloned')


In [ ]:
# CELL 4 — DOWNLOAD PRE-TRAINED UTILITY MODELS
import os
%cd /content/StyleTTS2
os.makedirs('Utils/ASR', exist_ok=True)
os.makedirs('Utils/JDC', exist_ok=True)
os.makedirs('Utils/PLBERT', exist_ok=True)

if not os.path.exists('Utils/ASR/epoch_00080.pth'):
    !wget -q --show-progress -O Utils/ASR/epoch_00080.pth "https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/epoch_00080.pth"

if not os.path.exists('Utils/JDC/bst.t7'):
    !wget -q --show-progress -O Utils/JDC/bst.t7 "https://github.com/nickoala/jdc/raw/master/bst.t7"

if not os.path.exists('Utils/PLBERT/config.json'):
    !git clone -q https://huggingface.co/yl4579/StyleTTS2-LibriTTS /tmp/s2pretrained
    !cp -r /tmp/s2pretrained/Utils/PLBERT Utils/

print('✅ Utility models downloaded')


In [ ]:
# CELL 5 — APPLY ALL CODE PATCHES
import re

# 1. Patch models.py & train_first.py for PyTorch 2.6+ compatibility
with open('/content/StyleTTS2/models.py', 'r') as f: code = f.read()
code = code.replace("torch.load(model_path, map_location='cpu')", "torch.load(model_path, map_location='cpu', weights_only=False)")
code = code.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)")
with open('/content/StyleTTS2/models.py', 'w') as f: f.write(code)

with open('/content/StyleTTS2/train_first.py', 'r') as f: code = f.read()
code = code.replace("torch.load(path, map_location='cpu')", "torch.load(path, map_location='cpu', weights_only=False)")
with open('/content/StyleTTS2/train_first.py', 'w') as f: f.write(code)

# 2. Patch hifigan.py for batch shape safety (Fix 1 & Fix 2)
with open('/content/StyleTTS2/Modules/hifigan.py', 'r') as f: code = f.read()
code = re.sub(r'([ \t]*)F0_curve = nn\.functional\.conv1d\(F0_curve\.unsqueeze\(1\)', r'\1if F0_curve.ndim == 3 and F0_curve.shape[-1] == 1:\n\1    F0_curve = F0_curve.squeeze(-1)\n\1if F0_curve.ndim == 1:\n\1    F0_curve = F0_curve.unsqueeze(0)\n\1F0_curve = nn.functional.conv1d(F0_curve.unsqueeze(1)', code)
code = re.sub(r'([ \t]*)F0 = self\.F0_conv\(F0_curve\.unsqueeze\(1\)\)', r'\1if F0_curve.ndim == 1:\n\1    F0_curve = F0_curve.unsqueeze(0)\n\1F0 = self.F0_conv(F0_curve.unsqueeze(1))', code)
with open('/content/StyleTTS2/Modules/hifigan.py', 'w') as f: f.write(code)

# 3. Patch models.py for batch shape safety (AdaLayerNorm)
with open('/content/StyleTTS2/models.py', 'r') as f: code = f.read()
code = re.sub(r'([ \t]*)h = h\.view\(h\.size\(0\), h\.size\(1\), 1\)', r'\1if h.ndim == 1:\n\1    h = h.unsqueeze(0)\n\1h = h.view(h.size(0), h.size(1), 1)', code)
with open('/content/StyleTTS2/models.py', 'w') as f: f.write(code)

# 4. Patch train_second.py for aggressive GPU garbage collection
with open('/content/StyleTTS2/train_second.py', 'r') as f: code = f.read()
old_val = "print('Epochs: %d\\nValidation loss: %.3f, Dur loss: %.3f, F0 loss: %.3f' % (epoch, val_loss, val_dur_loss, val_f0_loss))"
new_val = old_val + "\n        import gc; gc.collect(); torch.cuda.empty_cache()"
code = code.replace(old_val, new_val)
with open('/content/StyleTTS2/train_second.py', 'w') as f: f.write(code)

print('✅ StyleTTS2 codebase fully patched (All 5 fixes)!')


In [ ]:
# CELL 6 — UPLOAD YOUR AUDIO ZIP
from google.colab import files
import zipfile, os, glob

print('📁 Upload mcqueen_audio.zip when prompt appears...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
os.makedirs('/content/mcqueen_wavs', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/mcqueen_wavs')

wavs = glob.glob('/content/mcqueen_wavs/**/*.wav', recursive=True) + glob.glob('/content/mcqueen_wavs/*.wav')
print(f'✅ Extracted {len(wavs)} WAV files')


In [ ]:
# CELL 7 — AUTO-TRANSCRIBE WITH WHISPER
import whisper, json, os, glob, soundfile as sf, torch, gc

gc.collect()
torch.cuda.empty_cache()

model_whisper = whisper.load_model('base')
wavs = glob.glob('/content/mcqueen_wavs/**/*.wav', recursive=True) + glob.glob('/content/mcqueen_wavs/*.wav')

data = []
for wav_path in sorted(set(wavs)):
    res = model_whisper.transcribe(wav_path, language='en')
    txt = res['text'].strip()
    if not txt:
        continue
    info = sf.info(wav_path)
    data.append({'audio': wav_path, 'text': txt, 'duration': round(info.duration, 2)})
    print(f'[{len(data):02d}] {os.path.basename(wav_path)}: "{txt[:50]}"')

del model_whisper
gc.collect()
torch.cuda.empty_cache()

with open('/content/mcqueen_meta.json', 'w') as f:
    json.dump(data, f, indent=2)
print(f'\n✅ Transcribed {len(data)} clips, GPU memory cleared.')


In [ ]:
# CELL 8 — FILTER DATASET & BUILD FILELISTS
import json, os, random

with open('/content/mcqueen_meta.json') as f:
    data = json.load(f)

clean = [d for d in data if d['duration'] <= 8.0 and len(d['text']) < 120]
print(f'Dataset: {len(clean)} / {len(data)} clean clips retained.')

random.seed(42)
random.shuffle(clean)
split = max(1, int(len(clean) * 0.1))
val_data = clean[:split]
train_data = clean[split:]

os.makedirs('/content/StyleTTS2/Data/McQueen', exist_ok=True)

def write_filelist(items, path):
    with open(path, 'w') as f:
        for item in items:
            f.write(f"{item['audio']}|{item['text']}|0\n")

write_filelist(train_data, '/content/StyleTTS2/Data/McQueen/train_list.txt')
write_filelist(val_data,   '/content/StyleTTS2/Data/McQueen/val_list.txt')

print(f'✅ Train list: {len(train_data)} clips | Val list: {len(val_data)} clips')


In [ ]:
# CELL 9 — GENERATE STAGE 1 & STAGE 2 CONFIGS
import os, yaml

!wget -q -O /content/StyleTTS2/config_base.yml https://raw.githubusercontent.com/yl4579/StyleTTS2/main/Configs/config_ft.yml

with open('/content/StyleTTS2/config_base.yml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data_params']['train_data'] = 'Data/McQueen/train_list.txt'
cfg['data_params']['val_data'] = 'Data/McQueen/val_list.txt'
cfg['log_dir'] = 'Models/McQueen/'
cfg['batch_size'] = 2
cfg['max_len'] = 150
cfg['pretrained_model'] = ''
cfg['epochs_1st'] = 50
cfg['save_freq'] = 5

if 'loss_params' not in cfg:
    cfg['loss_params'] = {}
cfg['loss_params']['TMA_epoch'] = 50

with open('/content/StyleTTS2/config_stage1.yml', 'w') as f:
    yaml.dump(cfg, f)

cfg['max_epoch'] = 250
cfg['save_freq'] = 10
with open('/content/StyleTTS2/config_stage2.yml', 'w') as f:
    yaml.dump(cfg, f)

print('✅ config_stage1.yml and config_stage2.yml generated!')


In [ ]:
# CELL 10 — RUN STAGE 1 TRAINING (~35-45 mins)
import os, torch, gc
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%cd /content/StyleTTS2

!python train_first.py -p config_stage1.yml


In [ ]:
# CELL 10.5 — LINK STAGE 1 CHECKPOINT & DOWNLOAD BASE MODEL
import yaml, os, glob

print('⬇️ Downloading LibriTTS base model for Stage 2...')
os.makedirs('/content/StyleTTS2/Models/LibriTTS', exist_ok=True)
if not os.path.exists('/content/StyleTTS2/Models/LibriTTS/epochs_2nd_00020.pth'):
    !wget -q --show-progress -O /content/StyleTTS2/Models/LibriTTS/epochs_2nd_00020.pth "https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/epochs_2nd_00020.pth"

cfg_path = '/content/StyleTTS2/config_stage2.yml'
ckpts = sorted(glob.glob('/content/StyleTTS2/Models/McQueen/epoch_1st_*.pth'))
if not ckpts:
    print("❌ No Stage 1 checkpoints found! Stage 1 failed to train.")
else:
    latest = os.path.basename(ckpts[-1])
    print(f"🔗 Linking Stage 1 checkpoint: {latest}")
    with open(cfg_path, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['first_stage_path'] = latest
    cfg['pretrained_model'] = 'Models/LibriTTS/epochs_2nd_00020.pth'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f)
    print('✅ Stage 2 Configuration Ready!')


In [ ]:
# CELL 11 — RUN STAGE 2 TRAINING (250 EPOCHS, ~2.5 hrs)
import os, torch, gc
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%cd /content/StyleTTS2

!python train_second.py -p config_stage2.yml


In [ ]:
# CELL 12 — PACKAGE & DOWNLOAD FINISHED MODEL
import glob, re, torch, io, shutil, os
from google.colab import files

ckpts = glob.glob('/content/StyleTTS2/Models/McQueen/epoch_2nd_*.pth')
if not ckpts:
    print('❌ No Stage 2 checkpoint found yet')
else:
    latest = sorted(ckpts, key=lambda x: int(re.search(r'epoch_2nd_(\d+)', x).group(1)))[-1]
    ep = int(re.search(r'epoch_2nd_(\d+)', latest).group(1))
    print(f'Packaging epoch {ep} checkpoint...')

    full = torch.load(latest, map_location='cpu', weights_only=False)
    net = full.get('net', full)
    keep = ['bert', 'bert_encoder', 'predictor', 'text_encoder', 'decoder', 'diffusion']
    pruned = {k: v for k, v in net.items() if any(g in k for g in keep)}
    out_pth = '/content/mcqueen_model_pruned.pth'
    torch.save({'net': pruned}, out_pth)
    shutil.copy('/content/StyleTTS2/config_stage2.yml', '/content/config.yml')

    zip_name = f'/content/mcqueen_ep{ep}_trained.zip'
    os.system(f'zip -j {zip_name} {out_pth} /content/config.yml')
    size = os.path.getsize(zip_name) / 1e6
    print(f'✅ Ready: {zip_name} ({size:.0f} MB) — downloading...')
    files.download(zip_name)
